# 🧠 Single Agent Pipeline Project

## Problem Statement
Build a **Single-Agent Smart Assistant** that:
- Understands user queries
- Routes tasks based on intent
- Uses tools when required
- Returns structured JSON output

### The agent should handle:
- Math queries → Calculator Tool
- Keyword extraction → Keyword Tool
- General queries → Direct response

---
### 🛠️ What You Need to Implement
- Agent logic
- Conditional routing
- Tool integration
- Basic error handling

### 🚀 Bonus
- Improve routing
- Add logging
- Add more tools


In [ ]:
# 🛠️ TOOL 1: Calculator

def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        return str(eval(expression))
    except Exception:
        return "Error in calculation"

In [ ]:
# 🛠️ TOOL 2: Keyword Extractor

def extract_keywords(text: str) -> list:
    """Extract keywords from text."""
    try:
        words = text.split()
        keywords = list(set([w.lower() for w in words if len(w) > 4]))
        return keywords[:5]
    except Exception:
        return []

## 🚀 Bonus Tools

Additional tools to extend the agent's capabilities beyond the base requirements.

In [ ]:
# 🛠️ TOOL 3: Sentiment Analyzer (Bonus)

def analyze_sentiment(text: str) -> dict:
    """Perform basic rule-based sentiment analysis on the given text."""
    try:
        positive_words = {
            "good", "great", "excellent", "amazing", "wonderful", "fantastic",
            "happy", "love", "best", "awesome", "brilliant", "outstanding",
            "positive", "beautiful", "perfect", "enjoy", "liked", "helpful"
        }
        negative_words = {
            "bad", "terrible", "awful", "horrible", "worst", "hate",
            "poor", "ugly", "disappointing", "sad", "angry", "negative",
            "broken", "failed", "useless", "annoying", "boring", "dreadful"
        }
        words = text.lower().split()
        pos_count = sum(1 for w in words if w.strip('.,!?') in positive_words)
        neg_count = sum(1 for w in words if w.strip('.,!?') in negative_words)
        total = pos_count + neg_count

        if total == 0:
            sentiment = "neutral"
            confidence = 0.5
        elif pos_count > neg_count:
            sentiment = "positive"
            confidence = round(pos_count / total, 2)
        elif neg_count > pos_count:
            sentiment = "negative"
            confidence = round(neg_count / total, 2)
        else:
            sentiment = "mixed"
            confidence = 0.5

        return {
            "sentiment": sentiment,
            "confidence": confidence,
            "positive_count": pos_count,
            "negative_count": neg_count
        }
    except Exception:
        return {"sentiment": "unknown", "confidence": 0.0}


# 🛠️ TOOL 4: Text Summarizer (Bonus)

def summarize_text(text: str, max_sentences: int = 2) -> str:
    """Summarize text by extracting the most important sentences."""
    try:
        sentences = [s.strip() for s in text.replace('!', '.').replace('?', '.').split('.') if s.strip()]
        if len(sentences) <= max_sentences:
            return '. '.join(sentences) + '.'

        # Score sentences by word count and keyword density
        scored = []
        all_words = text.lower().split()
        word_freq = {}
        for w in all_words:
            w_clean = w.strip('.,!?;:')
            if len(w_clean) > 3:
                word_freq[w_clean] = word_freq.get(w_clean, 0) + 1

        for i, sentence in enumerate(sentences):
            words = sentence.lower().split()
            score = sum(word_freq.get(w.strip('.,!?;:'), 0) for w in words if len(w) > 3)
            # Boost first sentence (often contains key info)
            if i == 0:
                score *= 1.5
            scored.append((score, i, sentence))

        scored.sort(key=lambda x: x[0], reverse=True)
        # Pick top sentences, then re-order by original position
        top = sorted(scored[:max_sentences], key=lambda x: x[1])
        return '. '.join(s[2] for s in top) + '.'
    except Exception:
        return "Error in summarization"


# 🛠️ TOOL 5: Date & Time (Bonus)

from datetime import datetime

def get_datetime() -> dict:
    """Return the current date, time, and day of the week."""
    try:
        now = datetime.now()
        return {
            "date": now.strftime("%Y-%m-%d"),
            "time": now.strftime("%H:%M:%S"),
            "day": now.strftime("%A"),
            "timestamp": now.isoformat()
        }
    except Exception:
        return {"error": "Could not retrieve date/time"}

## 📋 Logging Setup (Bonus)

A simple logging utility to track every agent interaction for debugging and auditing.

In [ ]:
# 📋 Logging Setup (Bonus)

import logging

# Configure logger
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S"
)
logger = logging.getLogger("SmartAgent")
logger.setLevel(logging.DEBUG)

logger.info("Logger initialized for Smart Agent Pipeline.")

## 🤖 Implement Agent Logic Below

👉 Use conditional routing:
- If query contains "calculate" → use calculator
- If query contains "keywords" → use keyword extractor
- Else → general response

In [ ]:
# 🧭 Intent Classifier (Bonus: Improved Routing)

import re
import json

def classify_intent(query: str) -> str:
    """
    Classify the user's intent using keyword matching and pattern detection.
    Returns one of: 'calculation', 'keywords', 'sentiment', 'summarize',
                     'datetime', 'general'
    """
    q = query.lower().strip()

    # --- Calculation intent ---
    calc_triggers = ["calculate", "compute", "solve", "evaluate", "math", "add", "subtract", "multiply", "divide"]
    if any(trigger in q for trigger in calc_triggers):
        return "calculation"
    # Detect raw math expressions like "2 + 3", "100 / 5", etc.
    if re.search(r'\d+\s*[+\-*/^%]\s*\d+', q):
        return "calculation"

    # --- Keyword extraction intent ---
    kw_triggers = ["keyword", "extract", "key words", "important words", "key phrases"]
    if any(trigger in q for trigger in kw_triggers):
        return "keywords"

    # --- Sentiment analysis intent ---
    sent_triggers = ["sentiment", "feeling", "mood", "opinion", "positive or negative", "tone"]
    if any(trigger in q for trigger in sent_triggers):
        return "sentiment"

    # --- Summarization intent ---
    sum_triggers = ["summarize", "summary", "summarise", "brief", "shorten", "tldr", "tl;dr"]
    if any(trigger in q for trigger in sum_triggers):
        return "summarize"

    # --- Date/Time intent ---
    dt_triggers = ["time", "date", "today", "day", "clock", "now"]
    if any(trigger in q for trigger in dt_triggers):
        return "datetime"

    return "general"

In [ ]:
# 🔍 Expression Extractor Utility

def extract_math_expression(query: str) -> str:
    """
    Extract the mathematical expression from a natural-language query.
    E.g. 'Calculate 20 + 5' → '20 + 5'
    """
    # Try to find a math expression pattern
    match = re.search(r'([\d][\d\s+\-*/().^%]+[\d)])', query)
    if match:
        return match.group(1).strip()

    # Fallback: strip known command words and return the rest
    remove_words = ["calculate", "compute", "solve", "evaluate", "math", "what is", "what's"]
    result = query.lower()
    for word in remove_words:
        result = result.replace(word, "")
    return result.strip()


def extract_text_after_trigger(query: str, triggers: list) -> str:
    """
    Extract the text payload that comes after the trigger keyword.
    E.g. 'Extract keywords from Artificial Intelligence...' → 'Artificial Intelligence...'
    """
    q_lower = query.lower()
    for trigger in triggers:
        idx = q_lower.find(trigger)
        if idx != -1:
            after = query[idx + len(trigger):].strip()
            # Remove leading connectors like 'from', 'of', 'in', ':'
            after = re.sub(r'^(from|of|in|for|:)\s+', '', after, flags=re.IGNORECASE)
            if after:
                return after
    return query  # fallback to full query

In [ ]:
# 🤖 AGENT FUNCTION (IMPLEMENTED)

def agent(query: str) -> dict:
    """
    Single-Agent Smart Assistant.
    - Understands user queries via intent classification
    - Routes tasks to the appropriate tool
    - Returns structured JSON output
    """
    query_lower = query.lower().strip()
    logger.info(f"Received query: {query}")

    # ── Step 1: Classify Intent ──
    try:
        intent = classify_intent(query)
        logger.info(f"Classified intent: {intent}")
    except Exception as e:
        logger.error(f"Intent classification failed: {e}")
        return {"type": "error", "result": f"Intent classification error: {str(e)}"}

    # ── Step 2: Route to the appropriate tool ──
    try:
        # --- Calculation ---
        if intent == "calculation":
            expression = extract_math_expression(query)
            logger.debug(f"Extracted expression: {expression}")
            result = calculator(expression)
            logger.info(f"Calculator result: {result}")
            return {
                "type": "calculation",
                "input": expression,
                "result": result
            }

        # --- Keyword Extraction ---
        elif intent == "keywords":
            text = extract_text_after_trigger(
                query,
                ["keywords", "keyword", "extract", "key words", "important words"]
            )
            logger.debug(f"Text for keyword extraction: {text}")
            result = extract_keywords(text)
            logger.info(f"Extracted keywords: {result}")
            return {
                "type": "keywords",
                "input": text,
                "result": result
            }

        # --- Sentiment Analysis (Bonus) ---
        elif intent == "sentiment":
            text = extract_text_after_trigger(
                query,
                ["sentiment", "feeling", "mood", "opinion", "tone"]
            )
            logger.debug(f"Text for sentiment analysis: {text}")
            result = analyze_sentiment(text)
            logger.info(f"Sentiment result: {result}")
            return {
                "type": "sentiment",
                "input": text,
                "result": result
            }

        # --- Summarization (Bonus) ---
        elif intent == "summarize":
            text = extract_text_after_trigger(
                query,
                ["summarize", "summary", "summarise", "brief", "shorten", "tldr", "tl;dr"]
            )
            logger.debug(f"Text for summarization: {text}")
            result = summarize_text(text)
            logger.info(f"Summary result: {result}")
            return {
                "type": "summarize",
                "input": text,
                "result": result
            }

        # --- Date/Time (Bonus) ---
        elif intent == "datetime":
            result = get_datetime()
            logger.info(f"DateTime result: {result}")
            return {
                "type": "datetime",
                "result": result
            }

        # --- General / Fallback ---
        else:
            logger.info("Routed to general response.")
            return {
                "type": "general",
                "result": f"I understood your query: '{query}'. This is a general response - no specific tool was matched. Try asking me to calculate, extract keywords, analyze sentiment, summarize text, or check the date/time."
            }

    except Exception as e:
        logger.error(f"Error during tool execution: {e}")
        return {
            "type": "error",
            "result": f"An error occurred while processing your query: {str(e)}"
        }

## 📦 Expected Output Format

```
{
  "type": "calculation / keywords / general / error",
  "result": ...
}
```

In [ ]:
# 🧪 Test Cases

queries = [
    "Calculate 20 + 5",
    "Extract keywords from Artificial Intelligence is transforming industries",
    "What is machine learning?"
]

for q in queries:
    print("Query:", q)
    print("Response:", agent(q))
    print("-" * 50)

In [ ]:
# 🧪 Bonus Tool Test Cases

bonus_queries = [
    "Analyze the sentiment of This product is amazing and wonderful",
    "Summarize: Machine learning is a branch of artificial intelligence. It focuses on building systems that learn from data. These systems improve over time without being explicitly programmed. ML is used in recommendation engines, image recognition, and NLP.",
    "What is the date and time now?",
    "Compute 100 * 3 + 50",
    "What are the key words in Deep learning models require large datasets for training",
    "Tell me about Python programming"
]

for q in bonus_queries:
    print("Query:", q)
    result = agent(q)
    print("Response:", json.dumps(result, indent=2))
    print("=" * 60)

In [ ]:
# 🎯 Interactive Mode

while True:
    user_input = input("Enter query (type 'exit' to stop): ")
    if user_input.lower() == "exit":
        break
    print("Response:", agent(user_input))